<h3 style="color:#6FA8DC; font-weight:bold">Handling Missing Data → Categorical Variables</h3>

### Why categorical missing values are different
For numerical features we commonly use mean/median. For categorical features, these operations do not make sense.

Example:

```text
City
Delhi
Mumbai
NaN
Delhi
```

We need a **category-based** strategy.

### Main methods
1. Most Frequent / Mode
2. Missing Category
3. Domain-specific category
4. Missing Indicator when the fact that it was missing carries information


### 1. Most Frequent / Mode Imputation
Replace missing values with the most common category.

```python
SimpleImputer(strategy='most_frequent')
```

**Good when:** missing values are relatively few and the most frequent category is a reasonable substitute.

**Risk:** it increases the frequency of one category and can distort the distribution.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer

df = pd.read_csv('train(1).csv')

X = df[['Sex', 'Embarked', 'Cabin']].copy()
X.head()

In [ ]:
X.isnull().mean() * 100

In [ ]:
mode_imputer = SimpleImputer(strategy='most_frequent')

X_mode = pd.DataFrame(
    mode_imputer.fit_transform(X),
    columns=X.columns
)

X_mode.head()

### 2. Missing Category
Create a separate category such as `"Missing"` or `"Unknown"`.

```python
SimpleImputer(strategy='constant', fill_value='Missing')
```

This preserves the information that the original value was missing.

**Often useful when missingness itself may carry information.**

In [ ]:
missing_imputer = SimpleImputer(
    strategy='constant',
    fill_value='Missing'
)

X_missing = pd.DataFrame(
    missing_imputer.fit_transform(X),
    columns=X.columns
)

X_missing.head()

### 3. Domain-Based Imputation
Sometimes domain knowledge gives a meaningful replacement.

Example:
- Unknown city → `"Unknown"`
- Missing product type → `"Not Specified"`

Do not invent a category without understanding the business meaning.

### Modern ML way → ColumnTransformer + Pipeline
In a real model, categorical imputation should happen inside the preprocessing pipeline.

```text
Raw categorical data
        ↓
Categorical Imputer
        ↓
One-Hot Encoder
        ↓
Model
```

This keeps training and prediction preprocessing identical and prevents leakage when the pipeline is fitted correctly.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

X = df.drop(columns=['Survived'])
y = df['Survived']

categorical_cols = ['Sex', 'Embarked', 'Cabin']
numerical_cols = ['Age', 'SibSp', 'Parch', 'Fare', 'Pclass']

categorical_pipe = Pipeline([
    ('imputer', SimpleImputer(
        strategy='constant',
        fill_value='Missing'
    )),
    ('encoder', OneHotEncoder(
        handle_unknown='ignore'
    ))
])

numerical_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median'))
])

preprocessor = ColumnTransformer([
    ('num', numerical_pipe, numerical_cols),
    ('cat', categorical_pipe, categorical_cols)
])

model = Pipeline([
    ('preprocessing', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000))
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model.fit(X_train, y_train)
model.score(X_test, y_test)

### Important comparison

| Method | When |
|---|---|
| Most Frequent | Simple categorical missingness |
| Missing category | Missingness may itself be informative |
| Domain-specific | Strong business/domain knowledge |

### Important
Categorical data should **not** be passed to KNN/Iterative numerical imputation directly. First choose a strategy appropriate for categorical variables.

### Revision flow
```text
Categorical Missing Data
        ↓
Need to preserve "missing" information?
       / \
     YES  NO
      ↓    ↓
"Missing"  Mode
category   / domain
      ↓
OneHotEncoder
      ↓
Model
```